# Lesson 4: Chaining Prompts for Agentic Reasoning

## Automated Claim Triage: From First-Notice to the Right Queue

In this hands-on exercise, you will build a prompt chain that extracts key fields from free-form auto-claim reports, assesses damage severity, and routes each claim to one of several queues — `glass`, `fast_track`, `material_damage`, or `total_loss` — with code-based gate checks at every step.

## Outline:

- Setup
- Sample FNOL (First Notice of Loss) Texts
- Stage I: Information Extraction
- Stage II: Severity Assessment
- Stage III: Queue Routing
- Review Outputs

## Setup

Import necessary libraries and define helper functions, including a mock LLM client, code execution environment, and test runner.

In [1]:
# Import necessary libraries
# No changes needed in this cell
from openai import OpenAI  # For accessing the OpenAI API
from enum import Enum
import json
from pydantic import BaseModel, Field  # For structured data validation
from typing import List, Literal, Optional

In [3]:
# Set up LLM credentials
import os
from dotenv import load_dotenv
# Load environment variables from config/.env
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), "config", ".env"))
client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("OPENAI_API_KEY"), 
    )


In [4]:
import os
import time
from dataclasses import dataclass
from dotenv import load_dotenv

start_time = time.perf_counter()

load_dotenv()

SUPPORTED_MODELS = {
    "gpt-4o",
    "gpt-4o-mini",
    "gpt-4.1-mini",
    "gpt-4.1-nano",
}


@dataclass(frozen=True)
class Settings:
    api_key: str
    model: str
    max_tokens: int

    @staticmethod
    def load() -> "Settings":
        api_key = os.getenv("OPENAI_API_KEY", "").strip()
        model = os.getenv("OPENAI_MODEL", "gpt-4.1-nano").strip()
        max_tokens_raw = os.getenv("OPENAI_MAX_TOKENS", "1500").strip()

        if not api_key:
            raise ValueError("OPENAI_API_KEY is missing in .env")

        if model not in SUPPORTED_MODELS:
            raise ValueError(
                f"Invalid OPENAI_MODEL='{model}'. "
                f"Allowed values: {', '.join(sorted(SUPPORTED_MODELS))}"
            )

        try:
            max_tokens = int(max_tokens_raw)
        except ValueError:
            raise ValueError(
                f"Invalid OPENAI_MAX_TOKENS='{max_tokens_raw}'. It must be an integer."
            )

        return Settings(
            api_key=api_key,
            model=model,
            max_tokens=max_tokens,
        )

    @property
    def masked_api_key(self) -> str:
        """
        Shows only beginning and end of the API key.
        Example: sk-proj-abc...xyz123
        """
        if len(self.api_key) <= 10:
            return "*" * len(self.api_key)
        return f"{self.api_key[:8]}...{self.api_key[-6:]}"


settings = Settings.load()

API_KEY = settings.api_key
MODEL = settings.model
MAX_TOKENS = settings.max_tokens

# Your main code starts here
print("Running application...")
# Example: actual work
time.sleep(1.2)

end_time = time.perf_counter()
total_time = end_time - start_time

print("\n========== CONFIG / RUN SUMMARY ==========")
print(f"OpenAI API Key : {settings.masked_api_key}")
print(f"Model Selected : {MODEL}")
print(f"Max Tokens     : {MAX_TOKENS}")
print(f"Total Time     : {total_time:.2f} seconds")
print("==========================================")

Running application...

========== CONFIG / RUN SUMMARY ==========
OpenAI API Key : sk-proj-...IIazsA
Model Selected : gpt-4o
Max Tokens     : 1500
Total Time     : 1.20 seconds


## Sample FNOL (First Notice of Loss) Texts
Let's define a few sample First Notice of Loss (FNOL) texts to process through our chain.

In [5]:
# Define sample FNOL texts
# TODO: [Optional] Add more sample FNOL texts to test various scenarios

sample_fnols = [
    """
    Claim ID: C001
    Customer: John Smith
    Vehicle: 2018 Toyota Camry
    Incident: While driving on the highway, a rock hit my windshield and caused a small chip
    about the size of a quarter. No other damage was observed.
    """,
    """
    Claim ID: C002
    Customer: Sarah Johnson
    Vehicle: 2020 Honda Civic
    Incident: I was parked at the grocery store and returned to find someone had hit my car and
    dented the rear bumper and taillight. The taillight is broken and the bumper has a large dent.
    """,
    """
    Claim ID: C003
    Customer: Michael Rodriguez
    Vehicle: 2022 Ford F-150
    Incident: I was involved in a serious collision at an intersection. The front of my truck is
    severely damaged, including the hood, bumper, radiator, and engine compartment. The airbags
    deployed and the vehicle is not drivable.
    """,
    """
    Claim ID: C004
    Customer: Emma Williams
    Vehicle: 2019 Subaru Outback
    Incident: My car was damaged in a hailstorm. There are multiple dents on the hood, roof, and
    trunk. The side mirrors were also damaged and one window has a small crack.
    """,
    """
    Claim ID: C005
    Customer: David Brown
    Vehicle: 2021 Tesla Model 3
    Incident: Someone keyed my car in the parking lot. There are deep scratches along both doors
    on the driver's side.
    """,
    """
    Claim ID: C006
    Customer: Olivia Martinez
    Vehicle: 2017 Nissan Altima
    Incident: I backed into a concrete pole in a parking garage. The rear bumper is cracked and
    there is paint damage near the trunk edge.
    """,
    """
    Claim ID: C007
    Customer: James Anderson
    Vehicle: 2023 Chevrolet Traverse
    Incident: A deer ran into the road at night and I could not avoid it. The front grille,
    passenger-side headlight, and hood were damaged.
    """,
    """
    Claim ID: C008
    Customer: Sophia Taylor
    Vehicle: 2016 Hyundai Elantra
    Incident: During heavy rain, a tree branch fell onto the roof of my car. The roof is dented
    and the rear windshield shattered.
    """,
    """
    Claim ID: C009
    Customer: Daniel Thomas
    Vehicle: 2021 Jeep Grand Cherokee
    Incident: Another vehicle sideswiped my SUV while changing lanes. There are long scrape marks
    and dents along the driver-side doors and rear quarter panel.
    """,
    """
    Claim ID: C010
    Customer: Isabella Moore
    Vehicle: 2020 BMW X3
    Incident: I hit a large pothole on the freeway. The front passenger-side tire blew out and
    the wheel rim appears bent. The suspension also feels unstable.
    """,
    """
    Claim ID: C011
    Customer: William Jackson
    Vehicle: 2015 Ford Escape
    Incident: My vehicle was stolen overnight and recovered two days later. The ignition is
    damaged, the steering column is broken, and the interior has been vandalized.
    """,
    """
    Claim ID: C012
    Customer: Mia White
    Vehicle: 2022 Kia Sportage
    Incident: While stopped at a red light, another driver rear-ended me. The rear hatch will not
    close properly and the bumper and backup sensors are damaged.
    """,
    """
    Claim ID: C013
    Customer: Benjamin Harris
    Vehicle: 2019 Audi A4
    Incident: I scraped the passenger side of my car against a metal guardrail while avoiding
    debris. The doors have scratches and a visible dent near the front fender.
    """,
    """
    Claim ID: C014
    Customer: Charlotte Clark
    Vehicle: 2021 Mazda CX-5
    Incident: A shopping cart rolled into my parked vehicle during strong wind. It left a dent on
    the front passenger door and chipped the paint.
    """,
    """
    Claim ID: C015
    Customer: Lucas Lewis
    Vehicle: 2018 Ram 1500
    Incident: My truck slid on black ice and hit a roadside barrier. The front bumper, left
    fender, and axle area are damaged, and the truck pulls to one side.
    """,
    """
    Claim ID: C016
    Customer: Amelia Walker
    Vehicle: 2023 Mercedes-Benz C-Class
    Incident: Someone attempted to break into my car. The driver-side window was smashed and the
    door frame has visible pry marks and scratches.
    """,
    """
    Claim ID: C017
    Customer: Henry Hall
    Vehicle: 2017 Volkswagen Jetta
    Incident: I drove through a flooded street and the engine stalled shortly after. The vehicle
    will not start now, and water entered the cabin on the passenger side.
    """,
    """
    Claim ID: C018
    Customer: Evelyn Allen
    Vehicle: 2020 Lexus RX 350
    Incident: While entering my driveway, I misjudged the gate post and scraped the right side of
    the bumper and front quarter panel. There is paint transfer and minor denting.
    """,
    """
    Claim ID: C019
    Customer: Alexander Young
    Vehicle: 2022 Toyota RAV4
    Incident: Another car opened its door into mine in a crowded parking lot. The driver-side rear
    door has a sharp crease dent and chipped paint.
    """,
    """
    Claim ID: C020
    Customer: Harper King
    Vehicle: 2019 Chevrolet Malibu
    Incident: A loose piece of cargo fell from a truck ahead of me and struck the front of my car.
    The front bumper is cracked, the lower grille is broken, and there are scratches on the hood.
    """,
]

## Stage I: Information Extraction
In this stage, we'll create a prompt that extracts structured information from free-form FNOL text.

In [6]:
# Define a system prompt for information extraction according to the provided ClaimInformation class
# TODO: Complete the prompt by replacing the parts marked with **********


class ClaimInformation(BaseModel):
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[
        Literal[
            "windshield",
            "front",
            "rear",
            "side",
            "roof",
            "hood",
            "door",
            "bumper",
            "fender",
            "quarter panel",
            "trunk",
            "glass",
        ]
    ] = Field(..., min_items=1)


info_extraction_system_prompt = """
You are **********. Your task is to extract key information from First Notice of Loss (FNOL) reports.

Format your response as a valid JSON object with the following keys:
- claim_id (str): The claim ID
- **********
- **********
- etc.

********** <-- Extra instructions

Only respond with the JSON object, nothing else.
"""

C:\Users\jyotiremote\AppData\Local\Temp\ipykernel_14532\417084629.py:25: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  ] = Field(..., min_items=1)


In [7]:
# Define a gate check function and claim extraction function
# TODO: Complete the prompt by replacing the parts marked with **********


def gate1_validate_claim_info(claim_info_json: str) -> ClaimInformation:
    """
    Gate 1: Validates claim information extracted from FNOL text.
    Returns validated ClaimInformation object or raises validation error.
    """
    try:
        # Parse the JSON string
        claim_info_dict = json.loads(claim_info_json)
        # Validate with Pydantic model
        validated_info = ClaimInformation(**claim_info_dict)
        return validated_info
    except Exception as e:
        raise ValueError(f"Gate 1 validation failed: {str(e)}")


def extract_claim_info(fnol_text):
    """
    Stage 1: Extract structured information from FNOL text
    """
    messages = [
        {"role": "system", "content": info_extraction_system_prompt},
        {"role": "user", "content": fnol_text},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the extracted information
    try:
        validated_info = # ********** <-- Run the gate check on the response
        return validated_info
    except ValueError as e:
        print(f"Gate 1 failed: {e}")
        return None

SyntaxError: invalid syntax (1475647020.py, line 33)

In [ ]:
# Run the claim extraction function on the sample FNOLs
# No updates needed in this cell

extracted_claim_info_items = [
    extract_claim_info(fnol_text) for fnol_text in sample_fnols
]
extracted_claim_info_items

## Stage II: Severity Assessment
In this stage, we'll assess the severity of the damage based on the extracted information.

Note, our carrier applies the following heuristics:
- Minor damage: Small dents, scratches, glass chips (cost range: $100-$1,000)
- Moderate damage: Single panel damage, bumper replacement, door damage (cost range: $1,000-$5,000)
- Major damage: Structural damage, multiple panel replacement, engine/drivetrain issues, total loss candidates (cost range: $5,000-$50,000)

In this example we will let the LLM estimate the cost, though in production we would want a more accurate estimate, e.g. querying a database of repair costs.


In [ ]:
# Define a system prompt for severity assessment according to the provided SeverityAssessment class
# TODO: Complete the prompt by replacing the parts marked with **********


class SeverityAssessment(BaseModel):
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)


severity_assessment_system_prompt = """
You are an auto insurance damage assessor. Your task is to evaluate the severity of vehicle damage and estimate repair costs.

********** <-- Instructions

Only respond with the JSON object, nothing else.
"""

In [ ]:
# Define a gate check function and assess_severity function
# TODO: Complete the prompt by replacing the parts marked with **********


def gate2_cost_range_ok(severity_json: str) -> SeverityAssessment:
    """
    Gate 2: Validates that the estimated cost is within reasonable range for the severity.
    Returns validated SeverityAssessment object or raises validation error.
    """
    try:
        # Parse the JSON string
        severity_dict = json.loads(severity_json)
        # Validate with Pydantic model
        validated_severity = SeverityAssessment(**severity_dict)

        # Check cost range based on severity
        if (
            validated_severity.severity == "Minor"
            and (
                # ********** <-- est_cost outside of heuristic range for Minor will raise ValueError
            )
        ):
            raise ValueError(
                f"Minor damage should cost between $100-$1000, got ${validated_severity.est_cost}"
            )
        elif (
            validated_severity.severity == "Moderate"
            and (
                # ********** <-- est_cost outside of heuristic range for Moderate will raise ValueError
            )
        ):
            raise ValueError(
                f"Moderate damage should cost between $1000-$5000, got ${validated_severity.est_cost}"
            )
        elif (
            validated_severity.severity == "Major"
            and (
                # ********** <-- est_cost outside of heuristic range for Major will raise ValueError
            )
        ):
            raise ValueError(
                f"Major damage should cost between $5000-$50000, got ${validated_severity.est_cost}"
            )


        return validated_severity
    except Exception as e:
        raise ValueError(f"Gate 2 validation failed: {str(e)}")


def assess_severity(claim_info: ClaimInformation) -> Optional[SeverityAssessment]:
    """
    Stage 2: Assess severity based on damage description
    """

    # Convert Pydantic model to JSON string
    claim_info_json = claim_info.model_dump_json()

    messages = [
        {"role": "system", "content": severity_assessment_system_prompt},
        {"role": "user", "content": claim_info_json},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the severity assessment
    try:
        validated_severity = gate2_cost_range_ok(response)
        return validated_severity
    except ValueError as e:
        print(f"Gate 2 failed: {e}. Response: {response}")
        return None


In [ ]:
# Run the claim extraction function on the sample data
# No updates needed in this cell

severity_assessment_items = [
    assess_severity(item) for item in extracted_claim_info_items
]

severity_assessment_items

## 6. Stage III: Queue Routing
In this stage, we'll route the claim to the appropriate queue based on severity and damage area.

Use these routing rules:
- 'glass' queue: For Minor damage involving ONLY glass (windshield, windows)
- 'fast_track' queue: For other Minor damage
- 'material_damage' queue: For all Moderate damage
- 'total_loss' queue: For all Major damage

These are the priority levels:
- Priority 1 (highest): Safety issues, customer stranded
- Priority 2: Significant but contained damage, vehicle drivable
- Priority 3: Standard claims
- Priority 4: Minor issues, no mobility impact
- Priority 5 (lowest): Cosmetic only, no functional impact

In [ ]:
# Define a system prompt for claim routing according to the provided ClaimRouting class
# TODO: Complete the prompt by replacing the parts marked with **********


class ClaimRouting(BaseModel):
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss"]
    priority: int = Field(..., ge=1, le=5)


queue_routing_system_prompt = """
You are an auto insurance claim routing specialist. Your task is to determine the appropriate processing queue for each claim.

********** <-- Instructions

Only respond with the JSON object, nothing else.
"""

In [ ]:
# Define a gate check function and assess_severity function
# No updates needed in this cell


def gate3_validate_routing(routing_json: str) -> ClaimRouting:
    """
    Gate 3: Validates that the claim is routed to a valid queue.
    Returns validated ClaimRouting object or raises validation error.
    """
    try:
        # Parse the JSON string
        routing_dict = json.loads(routing_json)
        # Validate with Pydantic model
        validated_routing = ClaimRouting(**routing_dict)
        return validated_routing
    except Exception as e:
        raise ValueError(f"Gate 3 validation failed: {str(e)}")


def route_claim(
    claim_info: ClaimInformation, severity_assessment: Optional[SeverityAssessment]
) -> Optional[ClaimRouting]:
    """
    Stage 3: Route claim to appropriate queue
    """
    if severity_assessment is None:
        return None

    # Create input for the routing model
    routing_input = {
        "claim_info": claim_info.model_dump(),
        "severity_assessment": severity_assessment.model_dump(),
    }

    messages = [
        {"role": "system", "content": queue_routing_system_prompt},
        {"role": "user", "content": json.dumps(routing_input)},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the routing decision
    try:
        validated_routing = gate3_validate_routing(response)
        return validated_routing
    except ValueError as e:
        print(f"Gate 3 failed: {e}. Response: {response}")
        return None

In [ ]:
# Run the routing function on the sample data
# No updates needed in this cell

routed_claim_items = [
    route_claim(claim, severity_assessment)
    for claim, severity_assessment in zip(
        extracted_claim_info_items, severity_assessment_items
    )
]

routed_claim_items

## 7. Review Outputs

Let's put our data into a pandas dataframe for easier analysis.

In [ ]:
# No updates needed in this cell

import pandas as pd

records = []
for claim, severity_assessment, routed_claim in zip(
    extracted_claim_info_items, severity_assessment_items, routed_claim_items
):
    record = {}
    record.update(claim)
    record.update(severity_assessment)
    record.update(routed_claim)
    records.append(record)


# Show the entire dataframe since it is not too large
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(records)

df

## 8. Reflection & Transfer
Reflect on the effectiveness of chaining prompts for this task:

1. Prompt Chain Architecture:
* How does breaking down the task into stages affect the performance?
* What are the benefits of having gate checks between stages?
* How could the chain design be improved?

2. Error Handling:
* What types of errors might occur at each stage?
* How could we make the chain more robust against errors?
* What would a good fallback strategy look like?

3. Scalability:
* How well would this approach scale to handle more complex claims?
* What challenges might arise when processing thousands of claims?
* How could the prompt chain be optimized for efficiency?

4. Transfer to Other Domains:
* How could this prompt chaining approach be applied to other domains?
* What general principles of prompt chaining can we extract from this exercise?

## Summary

🎉 Congratulations! 🎉 You've built an impressive prompt chain system for insurance claims!
You transformed messy FNOL text into structured data, assessed damage severity, and routed claims to the right queues, all with robust gate checks! 🚀✨

Remember:

- 🔗 Chained prompts break complex tasks into manageable steps
- 🛡️ Gate checks prevent error cascades
- 🧠 Having specialized prompts helps keep code focused and maintainable

You've mastered a powerful pattern for countless business processes! 🏆
Amazing work on your agentic reasoning system! 💯